In [ ]:
"""
GRU4Rec with Playratio Weighting — 30Music Dataset
====================================================

Drop-in replacement for the S-KNN baseline. Keeps the same:
  - cfg dict pattern
  - temporal train/test split
  - strict + session evaluation metrics
  - MLflow logging

New signals used:
  - Track sequence ORDER (via GRU hidden state)
  - Playratio as a continuous loss weight (not just binary skip filter)
  - Optional cross-session user context (user embedding concat)

Hyperparameter tuning:
  - Set cfg["mode"] = "tune" and configure tuning params in cfg
  - Uses Optuna TPE sampler for efficient Bayesian search

Requirements
------------
    pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
    pip install mlflow pandas numpy tqdm optuna psutil

Run
---
    # Single training run:     set cfg["mode"] = "single"
    # Hyperparameter tuning:   set cfg["mode"] = "tune"
    python gru4rec_30music.py
"""

import os
import json
import time
import logging
import random
import platform
import subprocess
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import mlflow
import psutil

try:
    import optuna
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False

# ============================================================
# CONFIGURATION  (edit this dict to control everything)
# ============================================================

cfg = {
    # ---- Run mode ----
    "mode":                 "single",       # "single" = one training run
                                            # "tune"   = Optuna hyperparameter search

    # ---- Dataset ----
    "dataset_root":         "data/",
    "sample_sessions":      None,
    "sample_events":        200_000,

    # ---- Preprocessing ----
    "min_session_length":   3,
    "max_session_length":   100,
    "min_item_support":     5,
    "min_user_sessions":    3,
    "skip_ratio_threshold": 0.25,

    # ---- Model ----
    "model":                "gru4rec",
    "embedding_dim":        128,
    "hidden_dim":           256,
    "num_layers":           2,
    "dropout":              0.3,

    # ---- Training ----
    "epochs":               20,
    "batch_size":           512,
    "lr":                   1e-3,
    "weight_decay":         1e-5,
    "num_negatives":        20,
    "use_playratio_weight": True,
    "use_user_context":     False,
    "lr_step_size":         5,
    "lr_gamma":             0.5,

    # ---- Evaluation ----
    "top_n":                20,
    "test_fraction":        0.2,

    # ---- Hardware ----
    "device":               "cuda" if torch.cuda.is_available() else "cpu",
    "num_workers":          4,

    # ---- Optuna tuning (only used when mode="tune") ----
    "n_trials":             20,
    "study_name":           "gru4rec_tuning",

    # ---- MLflow ----
    "mlflow_tracking_uri":  "http://129.114.27.248:8000",
    "mlflow_experiment":    "30music-session-recommendation",
}

ENTITIES_DIR = os.path.join(cfg["dataset_root"], "entities")
RELATIONS_DIR = os.path.join(cfg["dataset_root"], "relations")

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)


# ============================================================
# ENVIRONMENT & COST TRACKING
# ============================================================

def collect_environment_info(device_str):
    """Collect comprehensive training environment information."""
    env = {}

    # --- System ---
    env["hostname"] = platform.node()
    env["os"] = f"{platform.system()} {platform.release()}"
    env["python_version"] = platform.python_version()
    env["cpu_model"] = platform.processor() or "unknown"
    env["cpu_count_logical"] = psutil.cpu_count(logical=True)
    env["cpu_count_physical"] = psutil.cpu_count(logical=False)
    env["ram_total_gb"] = round(psutil.virtual_memory().total / (1024 ** 3), 2)

    # --- PyTorch ---
    env["pytorch_version"] = torch.__version__
    env["cuda_available"] = torch.cuda.is_available()

    # --- GPU ---
    if torch.cuda.is_available():
        env["cuda_version"] = torch.version.cuda or "N/A"
        env["cudnn_version"] = str(torch.backends.cudnn.version()) if torch.backends.cudnn.is_available() else "N/A"
        env["gpu_count"] = torch.cuda.device_count()
        env["gpu_name"] = torch.cuda.get_device_name(0)
        gpu_props = torch.cuda.get_device_properties(0)
        env["gpu_memory_total_gb"] = round(gpu_props.total_mem / (1024 ** 3), 2)
        env["gpu_compute_capability"] = f"{gpu_props.major}.{gpu_props.minor}"
        env["gpu_multi_processor_count"] = gpu_props.multi_processor_count

        try:
            result = subprocess.run(
                ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                capture_output=True, text=True, timeout=5
            )
            env["nvidia_driver_version"] = result.stdout.strip().split("\n")[0]
        except Exception:
            env["nvidia_driver_version"] = "N/A"
    else:
        env["gpu_count"] = 0
        env["gpu_name"] = "N/A (CPU only)"

    # --- Git SHA ---
    try:
        result = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"],
            capture_output=True, text=True, timeout=5, cwd=os.path.dirname(os.path.abspath(__file__))
        )
        env["git_sha"] = result.stdout.strip() if result.returncode == 0 else "N/A"
    except Exception:
        env["git_sha"] = "N/A"

    return env


def log_environment_to_mlflow(env_info):
    """Log environment info as MLflow tags and params."""
    mlflow.set_tags({
        "env.hostname":         env_info.get("hostname", ""),
        "env.os":               env_info.get("os", ""),
        "env.gpu_name":         env_info.get("gpu_name", ""),
        "env.gpu_count":        str(env_info.get("gpu_count", 0)),
        "env.pytorch_version":  env_info.get("pytorch_version", ""),
        "env.cuda_version":     env_info.get("cuda_version", "N/A"),
        "env.git_sha":          env_info.get("git_sha", "N/A"),
    })

    mlflow.log_params({
        "env_gpu_name":         env_info.get("gpu_name", "N/A"),
        "env_gpu_memory_gb":    env_info.get("gpu_memory_total_gb", 0),
        "env_gpu_count":        env_info.get("gpu_count", 0),
        "env_cpu_count":        env_info.get("cpu_count_physical", 0),
        "env_ram_gb":           env_info.get("ram_total_gb", 0),
        "env_cuda_version":     env_info.get("cuda_version", "N/A"),
    })


def get_gpu_memory_stats():
    """Get current and peak GPU memory usage."""
    if not torch.cuda.is_available():
        return {"gpu_mem_allocated_mb": 0, "gpu_mem_reserved_mb": 0, "gpu_mem_peak_mb": 0}
    return {
        "gpu_mem_allocated_mb": round(torch.cuda.memory_allocated() / (1024 ** 2), 1),
        "gpu_mem_reserved_mb":  round(torch.cuda.memory_reserved() / (1024 ** 2), 1),
        "gpu_mem_peak_mb":      round(torch.cuda.max_memory_allocated() / (1024 ** 2), 1),
    }


# ============================================================
# PARSING
# ============================================================

def find_idomaar_file(directory, pattern):
    if not os.path.exists(directory):
        raise FileNotFoundError(f"Directory not found: {directory}")
    for fname in os.listdir(directory):
        if pattern.lower() in fname.lower() and fname.endswith(".idomaar"):
            return os.path.join(directory, fname)
    for fname in os.listdir(directory):
        if pattern.lower() in fname.lower():
            return os.path.join(directory, fname)
    raise FileNotFoundError(f"No file matching '{pattern}' in {directory}.")


def parse_sessions(filepath, cfg):
    max_rows = cfg["sample_sessions"]
    sessions = []
    parse_errors = 0

    with open(filepath, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows:
                break
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) < 4:
                parse_errors += 1
                continue

            session_id = parts[1]
            try:
                session_ts = int(parts[2])
            except (ValueError, TypeError):
                session_ts = 0

            raw_props  = parts[3] if len(parts) > 3 else ""
            raw_linked = parts[4] if len(parts) > 4 else ""
            user_id = None
            track_sequence = []
            linked = None

            if raw_linked:
                try:
                    linked = json.loads(raw_linked)
                except json.JSONDecodeError:
                    pass

            if linked is None and raw_props:
                brace_depth = 0
                split_pos = -1
                for ci, ch in enumerate(raw_props):
                    if ch == "{":
                        brace_depth += 1
                    elif ch == "}":
                        brace_depth -= 1
                        if brace_depth == 0 and ci < len(raw_props) - 1:
                            split_pos = ci + 1
                            break
                if split_pos > 0:
                    second_json = raw_props[split_pos:].strip()
                    if second_json:
                        try:
                            linked = json.loads(second_json)
                        except json.JSONDecodeError:
                            pass

            if linked is None:
                parse_errors += 1
                continue

            subjects = linked.get("subjects", [])
            if subjects:
                user_id = subjects[0].get("id")

            for obj in linked.get("objects", []):
                if obj.get("type") == "track":
                    track_sequence.append({
                        "track_id":  obj["id"],
                        "playstart": obj.get("playstart", 0),
                        "playtime":  obj.get("playtime", 0),
                        "playratio": obj.get("playratio"),
                        "action":    obj.get("action", "play"),
                    })

            track_sequence.sort(key=lambda x: x.get("playstart", 0))

            if user_id is not None and len(track_sequence) > 0:
                sessions.append({
                    "session_id": session_id,
                    "user_id":    user_id,
                    "timestamp":  session_ts,
                    "num_tracks": len(track_sequence),
                    "tracks":     track_sequence,
                })

    log.info(f"Parsed {len(sessions)} sessions ({parse_errors} parse errors)")
    return sessions


# ============================================================
# PREPROCESSING
# ============================================================

def sessions_to_dataframe(sessions, cfg):
    session_rows = []
    interaction_rows = []

    for s in sessions:
        session_rows.append({
            "session_id": s["session_id"],
            "user_id":    s["user_id"],
            "timestamp":  s["timestamp"],
            "num_tracks": s["num_tracks"],
        })
        for pos, t in enumerate(s["tracks"]):
            playratio = t.get("playratio")
            skipped = playratio is not None and playratio <= cfg["skip_ratio_threshold"]
            interaction_rows.append({
                "session_id": s["session_id"],
                "user_id":    s["user_id"],
                "position":   pos,
                "track_id":   t["track_id"],
                "playtime":   t.get("playtime", 0),
                "playratio":  playratio,
                "skipped":    skipped,
            })

    session_df     = pd.DataFrame(session_rows)
    interaction_df = pd.DataFrame(interaction_rows)

    session_df["timestamp"]    = pd.to_numeric(session_df["timestamp"], errors="coerce").astype("Int64")
    session_df["user_id"]      = pd.to_numeric(session_df["user_id"],   errors="coerce").astype("Int64")
    interaction_df["track_id"] = pd.to_numeric(interaction_df["track_id"], errors="coerce").astype("Int64")
    interaction_df["user_id"]  = pd.to_numeric(interaction_df["user_id"],  errors="coerce").astype("Int64")

    return session_df, interaction_df


def filter_data(session_df, interaction_df, cfg):
    log.info(f"Before filtering: {len(session_df)} sessions, {len(interaction_df)} interactions")

    engaged_df = interaction_df[~interaction_df["skipped"]].copy()

    session_lengths = engaged_df.groupby("session_id").size().reset_index(name="engaged_length")
    session_df = session_df.merge(session_lengths, on="session_id", how="left")
    session_df["engaged_length"] = session_df["engaged_length"].fillna(0).astype(int)

    valid_sessions = session_df[
        (session_df["engaged_length"] >= cfg["min_session_length"]) &
        (session_df["engaged_length"] <= cfg["max_session_length"])
    ]["session_id"]
    engaged_df = engaged_df[engaged_df["session_id"].isin(valid_sessions)]

    for _ in range(5):
        prev = len(engaged_df)
        item_counts  = engaged_df["track_id"].value_counts()
        valid_items  = item_counts[item_counts >= cfg["min_item_support"]].index
        engaged_df   = engaged_df[engaged_df["track_id"].isin(valid_items)]
        sess_lens    = engaged_df.groupby("session_id").size()
        valid_sids   = sess_lens[sess_lens >= cfg["min_session_length"]].index
        engaged_df   = engaged_df[engaged_df["session_id"].isin(valid_sids)]
        user_sess_c  = engaged_df.groupby("user_id")["session_id"].nunique()
        valid_users  = user_sess_c[user_sess_c >= cfg["min_user_sessions"]].index
        engaged_df   = engaged_df[engaged_df["user_id"].isin(valid_users)]
        if len(engaged_df) == prev:
            break

    session_df = session_df[session_df["session_id"].isin(engaged_df["session_id"].unique())]
    log.info(f"After filtering: {session_df['session_id'].nunique()} sessions, "
             f"{engaged_df['track_id'].nunique()} unique tracks, "
             f"{engaged_df['user_id'].nunique()} users, "
             f"{len(engaged_df)} interactions")

    return session_df, engaged_df


def build_vocabs(interaction_df):
    items = sorted(interaction_df["track_id"].unique())
    users = sorted(interaction_df["user_id"].unique())
    item2idx = {item: idx + 1 for idx, item in enumerate(items)}
    user2idx = {user: idx + 1 for idx, user in enumerate(users)}
    log.info(f"Vocab: {len(item2idx)} items, {len(user2idx)} users")
    return item2idx, user2idx


def build_sequences(interaction_df, item2idx, user2idx):
    sequences = []
    for session_id, group in interaction_df.sort_values("position").groupby("session_id"):
        user_id  = group["user_id"].iloc[0]
        item_ids = group["track_id"].tolist()
        ratios   = group["playratio"].tolist()

        item_idxs = [item2idx[i] for i in item_ids if i in item2idx]
        clean_ratios = [
            float(r) if (r is not None and not np.isnan(float(r))) else 1.0
            for r in ratios
        ]

        if len(item_idxs) >= 2:
            sequences.append({
                "session_id":  session_id,
                "user_idx":    user2idx.get(user_id, 0),
                "item_idxs":   item_idxs,
                "playratios":  clean_ratios,
            })
    return sequences


def temporal_split(session_df, sequences, cfg):
    session_df  = session_df.sort_values("timestamp")
    split_idx   = int(len(session_df) * (1 - cfg["test_fraction"]))
    train_ids   = set(session_df.iloc[:split_idx]["session_id"])
    test_ids    = set(session_df.iloc[split_idx:]["session_id"])

    train_seqs = [s for s in sequences if s["session_id"] in train_ids]
    test_seqs  = [s for s in sequences if s["session_id"] in test_ids]
    log.info(f"Train: {len(train_seqs)} sessions | Test: {len(test_seqs)} sessions")
    return train_seqs, test_seqs


# ============================================================
# DATASET
# ============================================================

class SessionDataset(Dataset):
    def __init__(self, sequences, num_items, num_negatives, use_playratio_weight):
        self.samples              = []
        self.num_items            = num_items
        self.num_negatives        = num_negatives
        self.use_playratio_weight = use_playratio_weight
        self._build(sequences)

    def _build(self, sequences):
        for seq in sequences:
            items  = seq["item_idxs"]
            ratios = seq["playratios"]
            user   = seq["user_idx"]
            for t in range(1, len(items)):
                prefix     = items[:t]
                next_item  = items[t]
                next_ratio = ratios[t] if self.use_playratio_weight else 1.0
                self.samples.append((prefix, user, next_item, next_ratio))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        prefix, user, next_item, ratio = self.samples[idx]
        negatives = random.sample(range(1, self.num_items + 1), self.num_negatives)
        return {
            "prefix":    torch.tensor(prefix,    dtype=torch.long),
            "user":      torch.tensor(user,      dtype=torch.long),
            "positive":  torch.tensor(next_item, dtype=torch.long),
            "negatives": torch.tensor(negatives, dtype=torch.long),
            "weight":    torch.tensor(ratio,     dtype=torch.float),
        }


def collate_fn(batch):
    max_len  = max(b["prefix"].size(0) for b in batch)
    prefixes = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, b in enumerate(batch):
        L = b["prefix"].size(0)
        prefixes[i, max_len - L:] = b["prefix"]

    return {
        "prefix":    prefixes,
        "user":      torch.stack([b["user"]      for b in batch]),
        "positive":  torch.stack([b["positive"]  for b in batch]),
        "negatives": torch.stack([b["negatives"] for b in batch]),
        "weight":    torch.stack([b["weight"]    for b in batch]),
    }


# ============================================================
# MODEL
# ============================================================

class GRU4Rec(nn.Module):
    def __init__(self, num_items, num_users, cfg):
        super().__init__()
        self.embedding_dim    = cfg["embedding_dim"]
        self.hidden_dim       = cfg["hidden_dim"]
        self.use_user_context = cfg["use_user_context"]

        self.item_emb = nn.Embedding(num_items + 1, cfg["embedding_dim"], padding_idx=0)

        gru_input_dim = cfg["embedding_dim"]
        if self.use_user_context:
            self.user_emb = nn.Embedding(num_users + 1, cfg["embedding_dim"], padding_idx=0)
            gru_input_dim += cfg["embedding_dim"]

        self.gru = nn.GRU(
            input_size=gru_input_dim,
            hidden_size=cfg["hidden_dim"],
            num_layers=cfg["num_layers"],
            batch_first=True,
            dropout=cfg["dropout"] if cfg["num_layers"] > 1 else 0,
        )
        self.dropout = nn.Dropout(cfg["dropout"])
        self.output_proj = nn.Linear(cfg["hidden_dim"], cfg["embedding_dim"])

    def encode_session(self, prefix_items, user_idxs):
        x = self.item_emb(prefix_items)

        if self.use_user_context:
            u = self.user_emb(user_idxs)
            u = u.unsqueeze(1).expand(-1, x.size(1), -1)
            x = torch.cat([x, u], dim=-1)

        lengths = (prefix_items != 0).sum(dim=1).cpu()
        lengths = lengths.clamp(min=1)

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )
        _, h_n = self.gru(packed)
        h_last = self.dropout(h_n[-1])
        session_repr = self.output_proj(h_last)
        return session_repr

    def forward(self, prefix_items, user_idxs, positive_items, negative_items):
        session_repr = self.encode_session(prefix_items, user_idxs)
        pos_emb = self.item_emb(positive_items)
        neg_emb = self.item_emb(negative_items)
        pos_scores = (session_repr * pos_emb).sum(dim=-1)
        neg_scores = torch.bmm(neg_emb, session_repr.unsqueeze(-1)).squeeze(-1)
        return pos_scores, neg_scores

    def predict_scores(self, prefix_items, user_idxs, all_item_embeddings):
        session_repr = self.encode_session(prefix_items, user_idxs)
        scores = session_repr @ all_item_embeddings[1:].T
        return scores


# ============================================================
# LOSS
# ============================================================

def bpr_max_loss(pos_scores, neg_scores, weights=None):
    neg_softmax  = torch.softmax(neg_scores, dim=-1)
    weighted_neg = (neg_softmax * neg_scores).sum(dim=-1)
    diff = pos_scores - weighted_neg
    bpr  = -torch.log(torch.sigmoid(diff) + 1e-8)
    if weights is not None:
        w = weights.clamp(min=0.05)
        bpr = bpr * w
    return bpr.mean()


# ============================================================
# TRAINING  (with per-epoch cost metrics)
# ============================================================

def train_epoch(model, loader, optimizer, device, run_cfg):
    model.train()
    total_loss = 0.0
    num_batches = 0
    total_samples = 0

    for batch in loader:
        prefix    = batch["prefix"].to(device)
        user      = batch["user"].to(device)
        positive  = batch["positive"].to(device)
        negatives = batch["negatives"].to(device)
        weight    = batch["weight"].to(device) if run_cfg["use_playratio_weight"] else None

        pos_scores, neg_scores = model(prefix, user, positive, negatives)
        loss = bpr_max_loss(pos_scores, neg_scores, weight)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss    += loss.item()
        num_batches   += 1
        total_samples += prefix.size(0)

    return total_loss / max(num_batches, 1), total_samples


# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(model, test_sequences, item2idx, run_cfg, device):
    model.eval()
    top_n        = run_cfg["top_n"]
    all_item_emb = model.item_emb.weight.to(device)

    strict_hits, strict_mrr   = 0, 0.0
    session_hits, session_mrr = 0, 0.0
    session_prec, session_rec = 0.0, 0.0
    all_recommended           = set()
    total                     = 0

    for seq in tqdm(test_sequences, desc="Evaluating", leave=False):
        items = seq["item_idxs"]
        user  = seq["user_idx"]

        for split_point in range(1, len(items)):
            prefix_items = items[:split_point]
            next_item    = items[split_point]
            remaining    = set(items[split_point:])

            prefix_t = torch.tensor([prefix_items], dtype=torch.long).to(device)
            user_t   = torch.tensor([user],         dtype=torch.long).to(device)

            scores = model.predict_scores(prefix_t, user_t, all_item_emb).squeeze(0)

            for seen in prefix_items:
                scores[seen - 1] = -1e9

            top_indices = torch.topk(scores, top_n).indices.cpu().numpy()
            predicted   = [int(i) + 1 for i in top_indices]
            all_recommended.update(predicted)
            total += 1

            if next_item in predicted:
                strict_hits += 1
                strict_mrr  += 1.0 / (predicted.index(next_item) + 1)

            hit_positions = [i for i, p in enumerate(predicted) if p in remaining]
            if hit_positions:
                session_hits += 1
                session_mrr  += 1.0 / (hit_positions[0] + 1)

            n_rel = len(hit_positions)
            session_prec += n_rel / top_n
            session_rec  += n_rel / len(remaining) if remaining else 0.0

    n = max(total, 1)
    return {
        "strict_HR":         strict_hits / n,
        "strict_MRR":        strict_mrr  / n,
        "session_HR":        session_hits / n,
        "session_MRR":       session_mrr  / n,
        "session_precision": session_prec / n,
        "session_recall":    session_rec  / n,
        "coverage":          len(all_recommended),
        "total_predictions": total,
    }


# ============================================================
# DATA PREPARATION (factored out for reuse in tuning)
# ============================================================

def prepare_data(cfg):
    """Parse, preprocess, build vocabs and sequences. Returns reusable dict."""
    sessions_file = find_idomaar_file(RELATIONS_DIR, "sessions")
    sessions      = parse_sessions(sessions_file, cfg)
    if not sessions:
        raise RuntimeError("No sessions parsed!")

    session_df, interaction_df = sessions_to_dataframe(sessions, cfg)
    session_df, interaction_df = filter_data(session_df, interaction_df, cfg)

    item2idx, user2idx = build_vocabs(interaction_df)
    sequences          = build_sequences(interaction_df, item2idx, user2idx)
    train_seqs, test_seqs = temporal_split(session_df, sequences, cfg)

    return {
        "item2idx":   item2idx,
        "user2idx":   user2idx,
        "train_seqs": train_seqs,
        "test_seqs":  test_seqs,
        "num_items":  len(item2idx),
        "num_users":  len(user2idx),
    }


# ============================================================
# SINGLE TRAINING RUN
# ============================================================

def run_training(run_cfg, data, env_info, is_tuning=False):
    """
    Execute one full training run. Returns final eval metrics dict.
    """
    num_items  = data["num_items"]
    num_users  = data["num_users"]
    train_seqs = data["train_seqs"]
    test_seqs  = data["test_seqs"]
    item2idx   = data["item2idx"]

    # ---- Dataset & DataLoader ----
    train_dataset = SessionDataset(
        train_seqs, num_items,
        run_cfg["num_negatives"],
        run_cfg["use_playratio_weight"],
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=run_cfg["batch_size"],
        shuffle=True,
        num_workers=run_cfg["num_workers"],
        collate_fn=collate_fn,
        pin_memory=(run_cfg["device"] == "cuda"),
    )

    # ---- Model ----
    device = torch.device(run_cfg["device"])
    model  = GRU4Rec(num_items, num_users, run_cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters())

    if not is_tuning:
        log.info(f"Training samples: {len(train_dataset)}")
        log.info(f"Parameters: {n_params:,}")

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=run_cfg["lr"],
        weight_decay=run_cfg["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=run_cfg["lr_step_size"],
        gamma=run_cfg["lr_gamma"],
    )

    # Reset peak GPU memory tracking
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # ---- Run name ----
    run_name = (
        f"gru4rec"
        f"_e{run_cfg['embedding_dim']}_h{run_cfg['hidden_dim']}_l{run_cfg['num_layers']}"
        f"{'_ratio' if run_cfg['use_playratio_weight'] else ''}"
        f"{'_user' if run_cfg['use_user_context'] else ''}"
        f"{'_tune' if is_tuning else ''}"
    )

    best_session_hr = 0.0
    final_results = {}

    with mlflow.start_run(run_name=run_name, nested=is_tuning) as run:
        # ---- Log config ----
        mlflow.log_params({k: str(v) for k, v in run_cfg.items()})
        mlflow.log_params({
            "num_items":        num_items,
            "num_users":        num_users,
            "train_sessions":   len(train_seqs),
            "test_sessions":    len(test_seqs),
            "train_samples":    len(train_dataset),
            "model_parameters": n_params,
        })
        mlflow.set_tags({
            "model_type":   "GRU4Rec",
            "dataset":      "30Music",
            "tuning_trial": str(is_tuning),
        })

        # ---- Log environment ----
        log_environment_to_mlflow(env_info)

        # ---- Training loop ----
        t_start = time.time()
        epoch_times = []

        for epoch in range(1, run_cfg["epochs"] + 1):
            t_epoch_start = time.time()

            avg_loss, samples_processed = train_epoch(
                model, train_loader, optimizer, device, run_cfg
            )
            scheduler.step()

            epoch_time = time.time() - t_epoch_start
            epoch_times.append(epoch_time)
            wall_time_so_far = time.time() - t_start

            # ---- Per-epoch cost metrics ----
            gpu_mem    = get_gpu_memory_stats()
            throughput = samples_processed / epoch_time if epoch_time > 0 else 0

            mlflow.log_metrics({
                "train_loss":                avg_loss,
                "epoch_time_sec":            round(epoch_time, 2),
                "wall_time_sec":             round(wall_time_so_far, 2),
                "throughput_samples_per_sec": round(throughput, 1),
                "learning_rate":             optimizer.param_groups[0]["lr"],
                "gpu_mem_allocated_mb":      gpu_mem["gpu_mem_allocated_mb"],
                "gpu_mem_peak_mb":           gpu_mem["gpu_mem_peak_mb"],
            }, step=epoch)

            if not is_tuning:
                log.info(
                    f"Epoch {epoch:02d}/{run_cfg['epochs']} | "
                    f"loss={avg_loss:.4f} | "
                    f"time={epoch_time:.1f}s | "
                    f"throughput={throughput:.0f} samples/s | "
                    f"peak_vram={gpu_mem['gpu_mem_peak_mb']:.0f}MB"
                )

            # ---- Evaluate every 5 epochs and at the end ----
            if epoch % 5 == 0 or epoch == run_cfg["epochs"]:
                t_eval_start = time.time()
                results = evaluate(model, test_seqs, item2idx, run_cfg, device)
                eval_time = time.time() - t_eval_start

                top_n = run_cfg["top_n"]
                mlflow.log_metrics({
                    f"strict_HR_at_{top_n}":        results["strict_HR"],
                    f"strict_MRR_at_{top_n}":       results["strict_MRR"],
                    f"session_HR_at_{top_n}":        results["session_HR"],
                    f"session_MRR_at_{top_n}":       results["session_MRR"],
                    f"session_precision_at_{top_n}": results["session_precision"],
                    f"session_recall_at_{top_n}":    results["session_recall"],
                    "coverage":                      results["coverage"],
                    "eval_time_sec":                 round(eval_time, 2),
                }, step=epoch)

                if not is_tuning:
                    log.info(
                        f"  Strict  HR@{top_n}={results['strict_HR']:.4f}  "
                        f"MRR={results['strict_MRR']:.4f}"
                    )
                    log.info(
                        f"  Session HR@{top_n}={results['session_HR']:.4f}  "
                        f"MRR={results['session_MRR']:.4f}  "
                        f"P={results['session_precision']:.4f}  "
                        f"R={results['session_recall']:.4f}"
                    )
                    log.info(f"  Eval time: {eval_time:.1f}s")

                if results["session_HR"] > best_session_hr:
                    best_session_hr = results["session_HR"]
                    if not is_tuning:
                        torch.save(model.state_dict(), "best_gru4rec.pt")
                        log.info(f"  → New best session HR: {best_session_hr:.4f} (saved)")

                final_results = results

        # ---- Final cost summary ----
        total_time = time.time() - t_start
        gpu_hours  = (total_time / 3600) * max(env_info.get("gpu_count", 1), 1) if env_info.get("cuda_available") else 0
        final_gpu_mem = get_gpu_memory_stats()

        mlflow.log_metrics({
            "total_train_seconds": round(total_time, 2),
            "total_train_minutes": round(total_time / 60, 2),
            "gpu_hours":           round(gpu_hours, 4),
            "avg_epoch_time_sec":  round(np.mean(epoch_times), 2),
            "std_epoch_time_sec":  round(np.std(epoch_times), 2),
            "min_epoch_time_sec":  round(np.min(epoch_times), 2),
            "max_epoch_time_sec":  round(np.max(epoch_times), 2),
            "peak_gpu_memory_mb":  final_gpu_mem["gpu_mem_peak_mb"],
            "best_session_HR":     best_session_hr,
        })

        if not is_tuning:
            log.info(f"\nTraining cost summary:")
            log.info(f"  Total time:     {total_time:.1f}s ({total_time/60:.1f} min)")
            log.info(f"  GPU hours:      {gpu_hours:.4f}")
            log.info(f"  Avg epoch time: {np.mean(epoch_times):.1f}s ± {np.std(epoch_times):.1f}s")
            log.info(f"  Peak VRAM:      {final_gpu_mem['gpu_mem_peak_mb']:.0f} MB")
            log.info(f"  Best session HR@{run_cfg['top_n']}: {best_session_hr:.4f}")
            log.info(f"  MLflow run ID:  {run.info.run_id}")

    final_results["best_session_HR"]     = best_session_hr
    final_results["total_train_seconds"] = total_time
    final_results["gpu_hours"]           = gpu_hours
    final_results["peak_gpu_memory_mb"]  = final_gpu_mem["gpu_mem_peak_mb"]
    return final_results


# ============================================================
# OPTUNA HYPERPARAMETER TUNING
# ============================================================

def create_optuna_trial_config(trial, base_cfg):
    """Define the Optuna search space. Returns a new cfg dict with sampled params."""
    trial_cfg = base_cfg.copy()

    # --- Architecture ---
    trial_cfg["embedding_dim"] = trial.suggest_categorical("embedding_dim", [64, 128, 256])
    trial_cfg["hidden_dim"]    = trial.suggest_categorical("hidden_dim", [128, 256, 512])
    trial_cfg["num_layers"]    = trial.suggest_int("num_layers", 1, 3)
    trial_cfg["dropout"]       = trial.suggest_float("dropout", 0.1, 0.5, step=0.1)

    # --- Training ---
    trial_cfg["lr"]            = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    trial_cfg["weight_decay"]  = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    trial_cfg["batch_size"]    = trial.suggest_categorical("batch_size", [256, 512, 1024])
    trial_cfg["num_negatives"] = trial.suggest_categorical("num_negatives", [10, 20, 50])

    # --- Feature flags ---
    trial_cfg["use_playratio_weight"] = trial.suggest_categorical("use_playratio_weight", [True, False])
    trial_cfg["use_user_context"]     = trial.suggest_categorical("use_user_context", [True, False])

    # --- LR schedule ---
    trial_cfg["lr_step_size"] = trial.suggest_int("lr_step_size", 3, 10)
    trial_cfg["lr_gamma"]     = trial.suggest_float("lr_gamma", 0.3, 0.8, step=0.1)

    # --- Fewer epochs during tuning for speed ---
    trial_cfg["epochs"] = trial.suggest_categorical("epochs", [10, 15, 20])

    return trial_cfg


def run_optuna_tuning(base_cfg, data, env_info):
    """Run Optuna TPE-based hyperparameter search."""
    if not OPTUNA_AVAILABLE:
        log.error("Optuna not installed. Run: pip install optuna")
        return

    n_trials   = base_cfg["n_trials"]
    study_name = base_cfg["study_name"]
    log.info(f"Starting Optuna tuning: {n_trials} trials, study={study_name}")

    def objective(trial):
        trial_cfg = create_optuna_trial_config(trial, base_cfg)
        log.info(f"\n{'='*60}")
        log.info(
            f"Trial {trial.number}: "
            f"e={trial_cfg['embedding_dim']} h={trial_cfg['hidden_dim']} "
            f"l={trial_cfg['num_layers']} d={trial_cfg['dropout']:.1f} "
            f"lr={trial_cfg['lr']:.5f} bs={trial_cfg['batch_size']}"
        )

        try:
            results = run_training(trial_cfg, data, env_info, is_tuning=True)
            session_hr = results.get("best_session_HR", 0.0)
            log.info(f"Trial {trial.number} → session_HR={session_hr:.4f}, time={results['total_train_seconds']:.0f}s")
            return session_hr
        except Exception as e:
            log.error(f"Trial {trial.number} failed: {e}")
            return 0.0

    # TPE sampler (Bayesian optimization)
    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(
        study_name=study_name,
        direction="maximize",
        sampler=sampler,
    )

    # Parent MLflow run for the sweep
    with mlflow.start_run(run_name=f"optuna_sweep_{study_name}"):
        mlflow.log_params({
            "tuning_n_trials":   n_trials,
            "tuning_sampler":    "TPE",
            "tuning_study_name": study_name,
        })
        mlflow.set_tags({"run_type": "hyperparameter_tuning"})
        log_environment_to_mlflow(env_info)

        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

        # ---- Log best results ----
        best = study.best_trial
        log.info(f"\n{'='*60}")
        log.info(f"Best trial: {best.number}")
        log.info(f"Best session_HR: {best.value:.4f}")
        log.info(f"Best params: {json.dumps(best.params, indent=2, default=str)}")

        mlflow.log_metrics({"best_trial_session_HR": best.value})
        mlflow.log_params({f"best_{k}": str(v) for k, v in best.params.items()})

        # Save study results as artifact
        trials_df = study.trials_dataframe()
        trials_path = "optuna_trials.csv"
        trials_df.to_csv(trials_path, index=False)
        mlflow.log_artifact(trials_path)

    return study


# ============================================================
# MAIN
# ============================================================

def main():
    log.info("=" * 60)
    log.info("[cfg] " + json.dumps(cfg, indent=2, default=str))
    log.info("=" * 60)

    # ---- Collect environment info ----
    env_info = collect_environment_info(cfg["device"])
    log.info(f"Device: {cfg['device']}")
    if env_info.get("cuda_available"):
        log.info(f"GPU: {env_info['gpu_name']} ({env_info['gpu_memory_total_gb']} GB)")
        log.info(f"CUDA: {env_info['cuda_version']} | cuDNN: {env_info['cudnn_version']}")
    log.info(f"CPU: {env_info['cpu_count_physical']} cores | RAM: {env_info['ram_total_gb']} GB")
    log.info(f"Git SHA: {env_info['git_sha']}")

    # ---- MLflow setup ----
    mlflow.set_tracking_uri(cfg["mlflow_tracking_uri"])
    mlflow.set_experiment(cfg["mlflow_experiment"])

    # ---- Prepare data (shared across all runs/trials) ----
    data = prepare_data(cfg)

    if cfg["mode"] == "tune":
        # ---- Hyperparameter tuning ----
        run_optuna_tuning(base_cfg=cfg, data=data, env_info=env_info)
    else:
        # ---- Single training run ----
        results = run_training(run_cfg=cfg, data=data, env_info=env_info, is_tuning=False)
        log.info(f"\nFinal best session HR@{cfg['top_n']}: {results['best_session_HR']:.4f}")

    log.info(f"MLflow UI: {cfg['mlflow_tracking_uri']}")


if __name__ == "__main__":
    main()

21:12:25 | ============================================================
21:12:25 | [cfg] {
  "dataset_root": "data/",
  "sample_sessions": null,
  "sample_events": 200000,
  "min_session_length": 3,
  "max_session_length": 100,
  "min_item_support": 5,
  "min_user_sessions": 3,
  "skip_ratio_threshold": 0.25,
  "model": "gru4rec",
  "embedding_dim": 128,
  "hidden_dim": 256,
  "num_layers": 2,
  "dropout": 0.3,
  "epochs": 20,
  "batch_size": 512,
  "lr": 0.001,
  "weight_decay": 1e-05,
  "num_negatives": 20,
  "use_playratio_weight": true,
  "use_user_context": false,
  "top_n": 20,
  "test_fraction": 0.2,
  "device": "cuda",
  "num_workers": 4,
  "mlflow_tracking_uri": "http://129.114.27.248:8000",
  "mlflow_experiment": "30music-session-recommendation"
}
21:12:25 | ============================================================
21:12:25 | Device: cuda


In [3]:
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
# !pip install mlflow pandas numpy tqdm